In [26]:
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from skimage.graph import MCP_Geometric
from scipy.ndimage import binary_closing, label, distance_transform_edt
from pyprojroot import here

In [27]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [28]:
ent = gpd.read_file(RAW_DATA_DIR/"geography/federal_states/00ent.shp")
cent = pd.read_csv(INTERIM_DATA_DIR/"municipality_centroids.csv")

In [29]:
land = ent.dissolve()  # single polygon/multipolygon, faster rasterize
land.geometry = land.geometry.buffer(0)  # fix invalid geoms
print(land.crs, land.total_bounds)

PROJCS["MEXICO_ITRF_2008_LCC",GEOGCS["ITRF2008",DATUM["International_Terrestrial_Reference_Frame_2008",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1061"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",12],PARAMETER["central_meridian",-102],PARAMETER["standard_parallel_1",17.5],PARAMETER["standard_parallel_2",29.5],PARAMETER["false_easting",2500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]] [ 911292.0295  319149.0625 4083063.0953 2349614.8717]


In [30]:
cent["CVEGEO"] = cent["CVEGEO"].astype(str).str.zfill(5)

cent_gdf = gpd.GeoDataFrame(
    cent, geometry=gpd.points_from_xy(cent.long, cent.lat), crs="EPSG:4326"
).to_crs(land.crs)

# Hēlax plant, Ciudad Ixtepec, Oaxaca
plant = gpd.GeoDataFrame(
    {"name": ["helax"]},
    geometry=gpd.points_from_xy([-95.0995], [16.5700]),  # confirm coords
    crs="EPSG:4326"
).to_crs(land.crs)

In [31]:
res = 1000  # meters
minx, miny, maxx, maxy = land.total_bounds
width  = int(np.ceil((maxx - minx) / res))
height = int(np.ceil((maxy - miny) / res))
transform = rasterio.transform.from_origin(minx, maxy, res, res)

mask = rasterize(
    [(geom, 1) for geom in land.geometry],
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8"
)

mask = binary_closing(mask, structure=np.ones((3,3)), iterations=1).astype(np.uint8)
cost = np.where(mask == 1, 1.0, np.inf).astype(np.float64)

In [32]:
def to_rc(x, y):
    r, c = rowcol(transform, x, y)
    return int(r), int(c)

def snap_to_land(r, c, mask, max_radius=100):
    if mask[r, c] == 1:
        return r, c
    for radius in range(1, max_radius + 1):
        rs = slice(max(r-radius,0), r+radius+1)
        cs = slice(max(c-radius,0), c+radius+1)
        sub = mask[rs, cs]
        if sub.any():
            ys, xs = np.where(sub == 1)
            dists = (ys + rs.start - r)**2 + (xs + cs.start - c)**2
            i = dists.argmin()
            return ys[i] + rs.start, xs[i] + cs.start
    raise ValueError(f"No land found within {max_radius}px for r={r}, c={c}")

In [33]:
cent_gdf["row"], cent_gdf["col"] = zip(*[
    snap_to_land(*to_rc(geom.x, geom.y), mask) for geom in cent_gdf.geometry
])
plant_r, plant_c = snap_to_land(*to_rc(plant.geometry.x[0], plant.geometry.y[0]), mask)

In [34]:
labeled, n = label(mask, structure=np.ones((3,3)))
component_sizes = np.bincount(labeled.ravel())
component_sizes[0] = 0
mainland_label = component_sizes.argmax()
mainland_mask = (labeled == mainland_label)
_, (nearest_r, nearest_c) = distance_transform_edt(~mainland_mask, return_indices=True)

def snap_to_mainland(r, c):
    if mainland_mask[r, c]:
        return r, c
    return nearest_r[r, c], nearest_c[r, c]

cent_gdf["row"], cent_gdf["col"] = zip(*[
    snap_to_mainland(r, c) for r, c in zip(cent_gdf["row"], cent_gdf["col"])
])
plant_r, plant_c = snap_to_mainland(plant_r, plant_c)

In [35]:
mcp = MCP_Geometric(cost, fully_connected=True)
costs, _ = mcp.find_costs([(plant_r, plant_c)])

In [36]:
cent_gdf["distance_ciit"] = costs[cent_gdf["row"], cent_gdf["col"]] * res / 1000

out = cent_gdf[["CVEGEO", "distance_ciit"]]

In [37]:
out.to_csv(INTERIM_DATA_DIR/"distance_to_ciit.csv", index=False)
print(out["distance_ciit"].max())
out.sort_values("distance_ciit").head()

3829.0860710365687


,CVEGEO,distance_ciit
1325,20014,6.414214
1179,20508,11.485281
1260,20030,13.899495
1443,20305,14.899495
1042,20005,18.556349
